# S&P 500 structured sustainability score

This notebook turns Bloomberg company data and a regulatory panel into two distinct outputs: a broad structured sustainability score and a net-zero transition score. Reusable mechanics live in `src/structured_esg.py`; the analytical choices remain visible here.


## 1. Setup

Set the input locations and make the scoring assumptions explicit.


In [ ]:
from pathlib import Path
from datetime import date
import json
import os
import sys
import warnings

# Load the numerical stack from the active environment first.
import numpy as np
import pandas as pd

# Some managed notebook environments provide openpyxl in a separate package path.
package_fallback = os.environ.get("NOTEBOOK_PYTHON_PACKAGES")
if package_fallback:
    sys.path.append(package_fallback)
import openpyxl

from IPython.display import display

# Allow execution from either the project root or notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
import structured_esg as esg

# Keep the executed notebook focused on results rather than library notices.
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)


In [ ]:
BLOOMBERG_PATH = esg.resolve_input(PROJECT_ROOT, "BLOOMBERG_ESG_XLSX", "ESGData.xlsx")
PANEL_PATH = esg.resolve_input(PROJECT_ROOT, "SP500_REGULATORY_PANEL_CSV", "sp500_esg_annual_features_2012_2024.csv")
OUTPUT_DIR = PROJECT_ROOT / "output" / "structured_score"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# These dates define the reporting windows used below.
QUANTITATIVE_ANCHOR_YEAR = 2024
GENERAL_FEATURES_TIME_BASIS = "latest/current workbook snapshot"
SCORING_AS_OF_DATE = date.today().isoformat()
TREND_YEARS = list(range(2019, 2025))
REGULATORY_YEARS = list(range(2020, 2025))
REGULATORY_RECENCY_WEIGHTS = {2020: 1, 2021: 2, 2022: 3, 2023: 4, 2024: 5}

# Small peer groups use the global distribution; incomplete blocks blend with 50.
MIN_PEER_COUNT = 15
MIN_REGULATORY_PEER_COUNT = 8
NEUTRAL_PRIOR = 50.0
ENVIRONMENT_SECTOR_MIX_LAMBDA = 0.50
REGULATORY_PENALTY_CAP = 15.0
CONFIDENCE_THRESHOLDS = {"High": 0.80, "Medium": 0.50}
SPARSE_COVERAGE_THRESHOLD = 0.10
HIGH_REDUNDANCY_SPEARMAN = 0.90

# Keep one stable sector order for matrices, tables, and figures.
SECTORS = [
    "Communication", "Consumer Discretionary", "Consumer Staples", "Energy",
    "Financials", "Health Care", "Industrials", "Information Technology",
    "Materials", "Real Estate", "Utilities",
]
MATERIALITY_VALUES = {"High": 1.0, "Medium": 0.5, "Low": 0.25, "N/A": 0.0}

print("Inputs:", BLOOMBERG_PATH.name, PANEL_PATH.name)
print("Output:", OUTPUT_DIR.relative_to(PROJECT_ROOT))


### Materiality and weights


In [ ]:
# Several features use the same sector pattern, so define each pattern once.
emissions_materiality = esg.sector_levels(SECTORS, "Medium", {
    "Energy": "High", "Materials": "High", "Utilities": "High", "Industrials": "High",
    "Financials": "Low", "Information Technology": "Low", "Communication": "Low"})
transition_commitment_materiality = esg.sector_levels(SECTORS, "Medium", {
    "Energy": "High", "Materials": "High", "Utilities": "High", "Industrials": "High"})
universal_high_materiality = esg.sector_levels(SECTORS, "High")
universal_medium_materiality = esg.sector_levels(SECTORS, "Medium")

FEATURE_MATERIALITY_LEVELS = {
    "scope12_reported_amount_2024": emissions_materiality.copy(),
    "scope12_employee_intensity_trend": emissions_materiality.copy(),
    "scope12_absolute_trend": emissions_materiality.copy(),
    "sbti_status": transition_commitment_materiality.copy(),
    "climate_governance_support": transition_commitment_materiality.copy(),
    "diversity": esg.sector_levels(SECTORS, "Medium", {
        "Communication": "High", "Consumer Discretionary": "High",
        "Consumer Staples": "High", "Financials": "High"}),
    "employee_safety": esg.sector_levels(SECTORS, "Low", {
        "Energy": "High", "Materials": "High", "Utilities": "High", "Industrials": "High",
        "Consumer Discretionary": "Medium", "Consumer Staples": "Medium", "Health Care": "Medium"}),
    "social_policy": universal_high_materiality.copy(),
    "employee_stability": universal_medium_materiality.copy(),
    "board_independence": universal_high_materiality.copy(),
    "ceo_separation": universal_high_materiality.copy(),
    "board_attendance": universal_high_materiality.copy(),
    "women_executives": universal_high_materiality.copy(),
    "sustainability_committee": universal_medium_materiality.copy(),
}

# Translate the readable labels into the multipliers used by the score.
FEATURE_MATERIALITY = {
    feature: {sector: MATERIALITY_VALUES[level] for sector, level in mapping.items()}
    for feature, mapping in FEATURE_MATERIALITY_LEVELS.items()
}


In [ ]:
# A source is scored only in sectors where its evidence is relevant.
REGULATORY_MATERIALITY_LEVELS = {
    "tri": esg.sector_levels(SECTORS, "N/A", {
        "Energy": "High", "Materials": "High", "Utilities": "High", "Industrials": "High",
        "Consumer Staples": "High", "Consumer Discretionary": "Medium",
        "Health Care": "Medium", "Information Technology": "Medium"}),
    "cfpb": esg.sector_levels(SECTORS, "N/A", {
        "Financials": "High", "Consumer Discretionary": "Medium", "Communication": "Low"}),
    "cpsc": esg.sector_levels(SECTORS, "N/A", {
        "Consumer Discretionary": "High", "Consumer Staples": "High", "Industrials": "Medium",
        "Information Technology": "Medium", "Health Care": "Low"}),
    "openfda": esg.sector_levels(SECTORS, "N/A", {
        "Health Care": "High", "Consumer Staples": "Medium", "Industrials": "Low"}),
}
REGULATORY_MATERIALITY = {
    source: {sector: MATERIALITY_VALUES[level] for sector, level in mapping.items()}
    for source, mapping in REGULATORY_MATERIALITY_LEVELS.items()
}


In [ ]:
# Retain the initial design so changes remain auditable.
ORIGINAL_FEATURE_WEIGHTS = {
    "Environmental": {
        "scope12_revenue_intensity": 0.40, "scope3_revenue_intensity": 0.20,
        "energy_revenue_intensity": 0.20, "resource_revenue_intensity": 0.10,
        "renewable_energy_ratio": 0.10},
    "Transition": {
        "scope12_employee_intensity_trend": 0.50, "sbti_status": 0.35,
        "climate_governance_support": 0.15},
    "Social": {
        "diversity": 0.35, "employee_safety": 0.30,
        "social_policy": 0.20, "employee_stability": 0.15},
    "Governance": {
        "board_independence": 0.35, "ceo_separation": 0.20, "board_attendance": 0.15,
        "women_executives": 0.15, "sustainability_committee": 0.15},
}

# The final environmental block uses the verified same-field footprint measure.
FINAL_FEATURE_WEIGHTS = {
    "Environmental": {"scope12_reported_amount_2024": 1.00},
    "Transition": ORIGINAL_FEATURE_WEIGHTS["Transition"].copy(),
    "Social": {
        "diversity": 0.40, "employee_safety": 0.20,
        "social_policy": 0.25, "employee_stability": 0.15},
    "Governance": ORIGINAL_FEATURE_WEIGHTS["Governance"].copy(),
}
FINAL_PILLAR_WEIGHTS = {
    "Environmental": 0.45, "Transition": 0.15, "Social": 0.20, "Governance": 0.20,
}

# Net-zero, commitment, and realized-transition scores remain separate diagnostics.
ORIGINAL_NET_ZERO_WEIGHTS = {
    "scope12_current_intensity": 0.25, "scope12_employee_intensity_trend": 0.25,
    "scope12_absolute_trend": 0.25, "sbti_status": 0.15,
    "climate_governance_support": 0.10,
}
FINAL_NET_ZERO_WEIGHTS = {
    "scope12_employee_intensity_trend": 1 / 3, "scope12_absolute_trend": 1 / 3,
    "sbti_status": 0.20, "climate_governance_support": 2 / 15,
}
COMMITMENT_WEIGHTS = {"sbti_status": 0.65, "climate_governance_support": 0.35}
REALIZED_TRANSITION_WEIGHTS = {
    "scope12_reported_amount_2024": 0.35,
    "scope12_employee_intensity_trend": 0.35,
    "tri_pollution_outcome": 0.30,
}
# Regulatory evidence can deduct at most 15 points; it is not a fifth pillar.
REGULATORY_SOURCE_WEIGHTS = {"tri": 0.40, "cfpb": 0.20, "cpsc": 0.20, "openfda": 0.20}

ASSESSMENT_DESCRIPTION = "Current structured sustainability assessment using a 2024 quantitative baseline and latest available governance and policy information."


## 2. Load the data

Read the three Bloomberg sheets and the annual regulatory panel. Structural checks stop execution only when the inputs cannot support the calculation.


In [ ]:
# data_only=True reads the values shown in the workbook rather than formula text.
bloomberg_wb = openpyxl.load_workbook(BLOOMBERG_PATH, read_only=True, data_only=True)

# Stop early only if a required structure is missing.
expected_sheets = {"General_Features", "Scope_Time_Series", "Financial_Time_Series"}
missing_sheets = expected_sheets - set(bloomberg_wb.sheetnames)
if missing_sheets:
    raise ValueError(f"Missing Bloomberg sheets: {sorted(missing_sheets)}")

general_security, general_diag = esg.read_bloomberg_sheet(bloomberg_wb["General_Features"])
scope_security, scope_diag = esg.read_bloomberg_sheet(bloomberg_wb["Scope_Time_Series"])
financial_security, financial_diag = esg.read_bloomberg_sheet(bloomberg_wb["Financial_Time_Series"])

panel = pd.read_csv(PANEL_PATH)
required_panel_columns = {
    "company_id", "observation_year", "primary_ticker", "constituent_tickers",
    "security_count", "company_name", "sector",
}
missing_columns = required_panel_columns - set(panel.columns)
if missing_columns:
    raise ValueError(f"Missing panel columns: {sorted(missing_columns)}")
if panel.duplicated(["company_id", "observation_year"]).any():
    raise ValueError("The panel contains duplicate company-year rows.")

# Use the latest annual row only to obtain one stable identity record per company.
companies = panel.sort_values(["company_id", "observation_year"])
companies = companies.drop_duplicates("company_id", keep="last")
company_columns = [
    "company_id", "primary_ticker", "constituent_tickers",
    "security_count", "company_name", "sector",
]
companies = companies[company_columns]
companies = companies.sort_values("company_id").reset_index(drop=True)


In [ ]:
# Expand share classes, map each Bloomberg security, and then collapse to companies.
security_map = esg.build_security_map(companies)
general_mapped, general_company, general_conflicts = esg.map_and_collapse(general_security, "General_Features", security_map)
_scope_mapped, scope_company, scope_conflicts = esg.map_and_collapse(scope_security, "Scope_Time_Series", security_map)
_financial_mapped, financial_company, financial_conflicts = esg.map_and_collapse(financial_security, "Financial_Time_Series", security_map)

# A company is collapsed only when its share classes contain the same values.
share_class_conflicts = general_conflicts + scope_conflicts + financial_conflicts
if share_class_conflicts:
    raise ValueError(f"Conflicting share-class values: {share_class_conflicts[:5]}")

# Record multiple-share-class companies for the validation report.
observed_dual = {}
for company_id, group in general_mapped.groupby("company_id"):
    if len(group) > 1:
        observed_dual[company_id] = set(group["normalized_ticker"])

# The workbook date is a practical snapshot date, not a field-level effective date.
workbook_modified = bloomberg_wb.properties.modified
if workbook_modified:
    GENERAL_FEATURES_AS_OF_DATE = workbook_modified.date().isoformat()
else:
    GENERAL_FEATURES_AS_OF_DATE = SCORING_AS_OF_DATE
GENERAL_FEATURES_AS_OF_DATE_BASIS = "workbook core modified-date proxy; not a field-effective date"

input_summary = {
    "Bloomberg securities": len(general_mapped),
    "Companies": len(companies),
    "Panel rows": len(panel),
    "Panel years": f"{panel.observation_year.min()}-{panel.observation_year.max()}",
    "Multiple-share-class companies": len(observed_dual),
}
print(f"Loaded {input_summary['Companies']} companies, {input_summary['Bloomberg securities']} securities, and {input_summary['Panel rows']:,} panel rows.")

# Save provenance now; calculation checks are collected once at the end.
validation = {
    "assessment_description": ASSESSMENT_DESCRIPTION,
    "input_sha256": {
        "bloomberg_workbook": esg.file_sha256(BLOOMBERG_PATH),
        "annual_panel": esg.file_sha256(PANEL_PATH)},
    "sheet_cleaning": {
        "General_Features": general_diag,
        "Scope_Time_Series": scope_diag,
        "Financial_Time_Series": financial_diag},
    "dual_share_classes": {key: sorted(value) for key, value in observed_dual.items()},
}


## 3. Build company features

Join the current company fields, derive 2019–2024 emissions trends, and convert categorical disclosures into numeric inputs.


In [ ]:
# Add Bloomberg security provenance to the stable company identities.
security_counts = general_mapped.groupby("company_id").size()
security_ids = general_mapped.groupby("company_id")["ID"].apply(lambda values: "|".join(sorted(values)))
identity = companies.copy()
identity["bloomberg_security_count_mapped"] = identity["company_id"].map(security_counts)
identity["bloomberg_security_ids"] = identity["company_id"].map(security_ids)

# Remove source-specific identity columns before joining the three Bloomberg tables.
identity_fields = {"ID", "name()", "id_isin()", "gics_sector_name()"}
general_fields = [column for column in general_security if column not in identity_fields]
scope_fields = [column for column in scope_security if column not in identity_fields]
financial_fields = [column for column in financial_security if column not in identity_fields]

company_sources = [
    (general_company, general_fields),
    (scope_company, scope_fields),
    (financial_company, financial_fields),
]
inputs = identity.copy()
for source_table, source_fields in company_sources:
    selected = esg.select_features(source_table, source_fields)
    inputs = inputs.merge(selected, on="company_id", how="left", validate="one_to_one")

# Convert only fields used as numbers; text disclosures remain unchanged.
numeric_fields = [
    column for column in inputs
    if column.startswith(("GHG_", "SALES_", "BS_", "EBITDA_", "CF_", "NUM_OF_"))
]
numeric_fields += [
    "ENERGY_CONSUMPTION", "RENEW_ENERGY_USE", "WATER_CONSUMPTION", "PCT_WATER_RECYCLED",
    "DISCHARGE_TO_WATER", "TOTAL_WASTE", "HAZARDOUS_WASTE", "WASTE_RECYCLED",
    "NUM_ENVIRON_FINES", "ENVIRON_FINES_AMT", "NUMBER_SPILLS", "AMOUNT_OF_SPILLS",
    "PCT_OF_GREEN_SUSTAIN_REVENUE", "WORK_ACCIDENTS_EMPLOYEES", "FATALITIES_CONTRACTORS",
    "FATALITIES_TOTAL", "EMPLOYEE_TRAINING_COST", "COMMUNITY_SPENDING", "EMPLOYEE_TURNOVER_PCT",
    "PCT_WOMEN_EMPLOYEES", "PCT_WOMEN_MGT", "FATALITIES_EMPLOYEES", "LOST_TIME_INCIDENT_RATE",
    "PCT_EMPLOYEES_UNIONIZED", "BOARD_SIZE", "PCT_INDEPENDENT_DIRECTORS",
    "AUDIT_CMTE_INDEPENDENCE_FLD_SCR", "SIZE_OF_AUDIT_COMMITTEE", "BOARD_AVERAGE_TENURE",
    "BOARD_AVERAGE_AGE", "PCT_OF_NON_EXEC_DIR_ON_BRD", "PCT_NON_EXEC_DIR_ON_AUD_CMTE",
    "PCT_NON_EXEC_DIR_ON_CMPNSTN_CMTE", "CHIEF_EXECUTIVE_OFFICER_TENURE",
    "PCT_OF_EXECUTIVES_THAT_ARE_WOMEN", "BOARD_DURATION", "BOARD_MEETINGS_PER_YR",
    "BOARD_MEETING_ATTENDANCE_PCT",
]
for column in sorted(set(numeric_fields).intersection(inputs.columns)):
    inputs[column] = pd.to_numeric(inputs[column], errors="coerce")


In [ ]:
# A Scope 1+2 amount exists only when both reported components are present.
inputs["scope12_reported_amount_2024"] = inputs[["GHG_SCOPE_1_2024", "GHG_SCOPE_2_2024"]].sum(axis=1, min_count=2)

# Keep one result row per company: absolute trend, count, intensity trend, count.
trend_rows = []
for row in inputs.itertuples(index=False):
    annual_amounts = []
    annual_intensities = []
    # Build comparable annual observations for this company.
    for year in TREND_YEARS:
        amount = esg.sum_if_complete(getattr(row, f"GHG_SCOPE_1_{year}"), getattr(row, f"GHG_SCOPE_2_{year}"))
        employees = getattr(row, f"NUM_OF_EMPLOYEES_{year}")
        annual_amounts.append(amount)
        intensity = amount / employees if pd.notna(amount) and pd.notna(employees) and employees > 0 else np.nan
        annual_intensities.append(intensity)

    # Four positive observations are required by the shared trend function.
    absolute_trend, absolute_n = esg.annualized_log_trend(TREND_YEARS, annual_amounts)
    intensity_trend, intensity_n = esg.annualized_log_trend(TREND_YEARS, annual_intensities)
    trend_rows.append((absolute_trend, absolute_n, intensity_trend, intensity_n))

trend_columns = ["scope12_absolute_trend_pct_per_year", "scope12_absolute_trend_observations",
                 "scope12_employee_intensity_trend_pct_per_year", "scope12_employee_intensity_trend_observations"]
inputs[trend_columns] = pd.DataFrame(trend_rows, index=inputs.index)

absolute_trends = inputs["scope12_absolute_trend_pct_per_year"]
intensity_trends = inputs["scope12_employee_intensity_trend_pct_per_year"]
inputs["emissions_transition_classification"] = [
    esg.transition_classification(absolute, intensity) for absolute, intensity in zip(absolute_trends, intensity_trends)
]


In [ ]:
# Missing disclosures remain missing; they are not treated as a negative answer.
inputs["sbti_status_raw"] = inputs["SBTI_NEAR_TERM_TARGET_STATUS"].map({"Targets Set": 2.0, "Committed": 1.0, "Removed": 0.0})
inputs["climate_governance_support_raw"] = esg.yn_numeric(inputs["CSR_SUSTAINABILITY_COMMITTEE"], "Y")
inputs["ceo_separation_raw"] = esg.yn_numeric(inputs["CEO_DUALITY"], "N")
inputs["sustainability_committee_raw"] = esg.yn_numeric(inputs["CSR_SUSTAINABILITY_COMMITTEE"], "Y")

# Average only the policy fields that a company actually reports.
policy_columns = [
    "UN_GLOBAL_COMPACT_SIGNATORY", "HUMAN_RIGHTS_POLICY", "EQUAL_OPPORTUNITY_POLICY",
    "FAIR_REMUNERATION_POLICY", "POLICY_AGAINST_CHILD_LABOR",
]
policy_numeric = pd.concat([esg.yn_numeric(inputs[column], "Y").rename(column) for column in policy_columns], axis=1)
inputs["social_policy_composite_raw"] = policy_numeric.mean(axis=1, skipna=True)
inputs["social_policy_fields_observed"] = policy_numeric.notna().sum(axis=1)
# Diversity uses the available workforce and management percentages.
diversity_columns = ["PCT_WOMEN_EMPLOYEES", "PCT_WOMEN_MGT"]
inputs["diversity_raw"] = inputs[diversity_columns].mean(axis=1, skipna=True)
inputs["diversity_fields_observed"] = inputs[diversity_columns].notna().sum(axis=1)

print("2024 Scope 1+2 coverage:", f"{inputs['scope12_reported_amount_2024'].notna().mean():.1%}")
print("Comparable absolute trends:", inputs["scope12_absolute_trend_pct_per_year"].notna().sum())
print("Comparable intensity trends:", inputs["scope12_employee_intensity_trend_pct_per_year"].notna().sum())


In [ ]:
# Candidate ratios stay in the audit even when unit checks prevent their use.
sales_available = inputs["SALES_REV_TURN_2024"].gt(0)
potential_conditions = {
    "scope12_revenue_intensity": inputs["scope12_reported_amount_2024"].notna() & sales_available,
    "scope3_revenue_intensity": inputs["GHG_SCOPE_3_2024"].notna() & sales_available,
    "energy_revenue_intensity": inputs["ENERGY_CONSUMPTION"].notna() & sales_available,
    "resource_revenue_intensity": inputs[["WATER_CONSUMPTION", "TOTAL_WASTE"]].notna().any(axis=1) & sales_available,
    "renewable_energy_ratio": inputs["RENEW_ENERGY_USE"].notna() & inputs["ENERGY_CONSUMPTION"].gt(0),
}

# These are the enabled raw features reviewed for coverage and redundancy.
audit_series = {
    "scope12_reported_amount_2024": inputs["scope12_reported_amount_2024"],
    "scope12_employee_intensity_trend": inputs["scope12_employee_intensity_trend_pct_per_year"],
    "scope12_absolute_trend": inputs["scope12_absolute_trend_pct_per_year"],
    "sbti_status": inputs["sbti_status_raw"],
    "climate_governance_support": inputs["climate_governance_support_raw"],
    "diversity": inputs["diversity_raw"],
    "employee_safety": inputs[["WORK_ACCIDENTS_EMPLOYEES", "FATALITIES_EMPLOYEES"]].mean(axis=1, skipna=True),
    "social_policy": inputs["social_policy_composite_raw"],
    "employee_stability": inputs["EMPLOYEE_TURNOVER_PCT"],
    "board_independence": inputs["PCT_INDEPENDENT_DIRECTORS"],
    "ceo_separation": inputs["ceo_separation_raw"],
    "board_attendance": inputs["BOARD_MEETING_ATTENDANCE_PCT"],
    "women_executives": inputs["PCT_OF_EXECUTIVES_THAT_ARE_WOMEN"],
    "sustainability_committee": inputs["sustainability_committee_raw"],
}

catalog = esg.feature_catalog(GENERAL_FEATURES_TIME_BASIS)
feature_dictionary, correlations, redundant_pairs, feature_coverage_by_sector = esg.audit_feature_catalog(
    inputs, catalog, audit_series, potential_conditions,
    ORIGINAL_FEATURE_WEIGHTS, FINAL_FEATURE_WEIGHTS,
    SPARSE_COVERAGE_THRESHOLD, HIGH_REDUNDANCY_SPEARMAN)
print(f"Feature audit complete: {len(feature_dictionary)} candidate definitions, {len(redundant_pairs)} high-correlation pairs.")


In [ ]:
# Each entry states the raw column and whether a larger value is better.
feature_score_specs = {
    "scope12_reported_amount_2024": ("scope12_reported_amount_2024", False),
    "scope12_employee_intensity_trend": ("scope12_employee_intensity_trend_pct_per_year", False),
    "scope12_absolute_trend": ("scope12_absolute_trend_pct_per_year", False),
    "sbti_status": ("sbti_status_raw", True),
    "climate_governance_support": ("climate_governance_support_raw", True),
    "diversity": ("diversity_raw", True),
    "work_accidents": ("WORK_ACCIDENTS_EMPLOYEES", False),
    "employee_fatalities": ("FATALITIES_EMPLOYEES", False),
    "social_policy": ("social_policy_composite_raw", True),
    "employee_stability": ("EMPLOYEE_TURNOVER_PCT", False),
    "board_independence": ("PCT_INDEPENDENT_DIRECTORS", True),
    "ceo_separation": ("ceo_separation_raw", True),
    "board_attendance": ("BOARD_MEETING_ATTENDANCE_PCT", True),
    "women_executives": ("PCT_OF_EXECUTIVES_THAT_ARE_WOMEN", True),
    "sustainability_committee": ("sustainability_committee_raw", True),
}

# Rank against sector peers when possible; the helper records every global fallback.
for feature, (raw_column, beneficial) in feature_score_specs.items():
    ranked = esg.percentile_good_scores(inputs, raw_column, beneficial, MIN_PEER_COUNT)
    for result_column in ranked.columns:
        inputs[f"{feature}_{result_column}"] = ranked[result_column]
    inputs[f"{feature}_score"] = ranked["sector_score"]

# Only the environmental footprint combines a sector score and a global score.
sector_component = ENVIRONMENT_SECTOR_MIX_LAMBDA * inputs["scope12_reported_amount_2024_sector_score"]
global_component = (1 - ENVIRONMENT_SECTOR_MIX_LAMBDA) * inputs["scope12_reported_amount_2024_global_score"]
inputs["scope12_reported_amount_2024_score"] = sector_component + global_component

# Safety is the average of the available accident and fatality evidence.
safety_columns = ["work_accidents_score", "employee_fatalities_score"]
inputs["employee_safety_score"] = inputs[safety_columns].mean(axis=1, skipna=True)
inputs["employee_safety_peer_group"] = np.where(
    inputs[safety_columns].notna().any(axis=1),
    "composite of observed sector/fallback percentiles",
    pd.NA)
safety_peer_counts = ["work_accidents_peer_count", "employee_fatalities_peer_count"]
inputs["employee_safety_peer_count"] = inputs[safety_peer_counts].min(axis=1, skipna=True).astype("Int64")

# Summarize fallbacks in a loop so each field is easy to trace.
peer_rows = []
for peer_group_column in [column for column in inputs if column.endswith("_peer_group")]:
    feature = peer_group_column.removesuffix("_peer_group")
    peer_count_column = peer_group_column.replace("_peer_group", "_peer_count")
    minimum_peer_count = None
    if inputs[peer_count_column].notna().any():
        minimum_peer_count = int(inputs[peer_count_column].min())
    peer_rows.append({
        "feature": feature,
        "observed": int(inputs[peer_group_column].notna().sum()),
        "global_fallback_count": int(inputs[peer_group_column].eq("global fallback").sum()),
        "minimum_peer_count_used": minimum_peer_count})
peer_diagnostics = pd.DataFrame(peer_rows)
print(f"Peer diagnostics recorded for {len(peer_diagnostics)} normalized features.")


## 4. Calculate the scores

Each block reweights its observed, material features and adjusts for missing evidence with:

`adjusted score = raw score × coverage + 50 × (1 − coverage)`

One shared scoring function is used for the pillars, transition diagnostics, and sensitivity tests.


In [ ]:
PILLAR_FEATURE_KEYS = {
    "environmental": "Environmental", "transition": "Transition",
    "social": "Social", "governance": "Governance",
}
# Every score block uses the same coverage adjustment and confidence rules.
block_options = {
    "materiality": FEATURE_MATERIALITY,
    "neutral_prior": NEUTRAL_PRIOR,
    "confidence_thresholds": CONFIDENCE_THRESHOLDS,
}

# Calculate E, T, S, and G with one parameterized scoring function.
for prefix, title in PILLAR_FEATURE_KEYS.items():
    result = esg.score_block(inputs, FINAL_FEATURE_WEIGHTS[title], **block_options)
    for column in result:
        inputs[f"{prefix}_{column}"] = result[column]

# Blend adjusted pillars only after the coverage adjustment has been applied.
inputs["structured_score_before_regulatory_penalty"] = 0.0
for prefix, title in PILLAR_FEATURE_KEYS.items():
    inputs["structured_score_before_regulatory_penalty"] += FINAL_PILLAR_WEIGHTS[title] * inputs[f"{prefix}_adjusted"]

# Reuse the same function for the two transition-related diagnostics.
for output_prefix, weights in {
    "net_zero_transition": FINAL_NET_ZERO_WEIGHTS,
    "commitment": COMMITMENT_WEIGHTS,
}.items():
    result = esg.score_block(inputs, weights, **block_options)
    inputs[f"{output_prefix}_raw_score"] = result["raw"]
    inputs[f"{output_prefix}_score"] = result["adjusted"]
    inputs[f"{output_prefix}_observed_weight"] = result["observed_weight"]
    inputs[f"{output_prefix}_confidence"] = result["confidence"]

pillar_summary = pd.DataFrame([
    {
        "pillar": title,
        "configured_weight": FINAL_PILLAR_WEIGHTS[title],
        "raw_min": inputs[f"{prefix}_raw"].min(),
        "raw_max": inputs[f"{prefix}_raw"].max(),
        "adjusted_min": inputs[f"{prefix}_adjusted"].min(),
        "adjusted_max": inputs[f"{prefix}_adjusted"].max(),
        "mean_observed_weight": inputs[f"{prefix}_observed_weight"].mean(),
        "low_confidence_companies": inputs[f"{prefix}_confidence"].eq("Low").sum(),
    }
    for prefix, title in PILLAR_FEATURE_KEYS.items()])
print("Pillar scores calculated for Environmental, Transition, Social, and Governance.")


In [ ]:
# Keep the five most recent years and assign more weight to later observations.
recent_panel = panel[panel["observation_year"].isin(REGULATORY_YEARS)].copy()
recent_panel["recency_weight"] = recent_panel["observation_year"].map(REGULATORY_RECENCY_WEIGHTS)

# Each source has one metric, one match flag, and an optional eligibility rule.
reg_specs = {
    "tri": ("tri_total_releases_lb_per_facility", "tri_match_flag", None),
    "cfpb": (
        "cfpb_not_timely_response_rate",
        "cfpb_match_flag",
        lambda group: group["cfpb_complaints"].fillna(0).ge(10)),
    "cpsc": ("cpsc_named_recall_count", "cpsc_match_flag", None),
    "openfda": ("openfda_enforcement_event_count", "openfda_match_flag", None),
}

# Aggregate each source separately, then join the four compact tables by company.
reg_aggregates = companies[["company_id"]].copy()
for source, (metric, flag, eligibility) in reg_specs.items():
    grouped = recent_panel.groupby("company_id", sort=False)
    aggregate = grouped.apply(lambda group: esg.recency_weighted_metric(group, metric, flag, eligibility), include_groups=False)
    aggregate = aggregate.rename(columns={
        "value": f"{source}_recent_value",
        "years_observed": f"{source}_recent_years_observed",
        "latest_observed_year": f"{source}_latest_observed_year"})
    aggregate = aggregate.reset_index()
    reg_aggregates = reg_aggregates.merge(aggregate, on="company_id", how="left", validate="one_to_one")

inputs = inputs.merge(reg_aggregates, on="company_id", how="left", validate="one_to_one")

# A high good-score means less adverse evidence; the penalty uses its inverse.
for source in reg_specs:
    value_column = f"{source}_recent_value"
    ranked = esg.percentile_good_scores(inputs, value_column, beneficial=False, min_peers=MIN_REGULATORY_PEER_COUNT)
    inputs[f"{source}_regulatory_good_score"] = ranked["sector_score"]
    inputs[f"{source}_regulatory_adverse_score"] = 100 - ranked["sector_score"]
    inputs[f"{source}_regulatory_peer_group"] = ranked["peer_group"]
    inputs[f"{source}_regulatory_peer_count"] = ranked["peer_count"]
    inputs[f"{source}_regulatory_peer_fallback_used"] = ranked["fallback_used"]


In [ ]:
# Combine only observed sources that are material to each company's sector.
penalty = esg.regulatory_penalty(inputs, REGULATORY_SOURCE_WEIGHTS, REGULATORY_MATERIALITY, REGULATORY_PENALTY_CAP)
inputs["regulatory_evidence_penalty_provisional"] = penalty["penalty"]
inputs["regulatory_penalty_observed_weight"] = penalty["observed_weight"]
inputs["regulatory_evidence_status"] = penalty["status"]
score_after_deduction = inputs["structured_score_before_regulatory_penalty"] - inputs["regulatory_evidence_penalty_provisional"]
inputs["structured_score_after_regulatory_penalty"] = np.maximum(0, score_after_deduction)
# Measure how much the provisional deduction changes the ranking.
inputs["rank_before_regulatory"] = inputs["structured_score_before_regulatory_penalty"].rank(method="min", ascending=False).astype(int)
inputs["rank_after_regulatory"] = inputs["structured_score_after_regulatory_penalty"].rank(method="min", ascending=False).astype(int)
inputs["rank_delta"] = inputs["rank_before_regulatory"] - inputs["rank_after_regulatory"]

# Regulatory evidence is a deduction and a separate credibility diagnostic, not a pillar.
inputs["tri_pollution_outcome_score"] = inputs["tri_regulatory_good_score"]
realized = esg.score_block(inputs, REALIZED_TRANSITION_WEIGHTS, **block_options)
inputs["realized_transition_score"] = realized["adjusted"]
inputs["realized_transition_observed_weight"] = realized["observed_weight"]
inputs["realized_transition_confidence"] = realized["confidence"]
inputs["credibility_gap"] = inputs["commitment_score"] - inputs["realized_transition_score"]

# Retain the largest penalties and rank movements for inspection and plotting.
largest_penalties = inputs.nlargest(12, "regulatory_evidence_penalty_provisional")[[
    "primary_ticker", "company_name", "sector", "regulatory_evidence_penalty_provisional",
    "regulatory_penalty_observed_weight", "tri_recent_value", "cfpb_recent_value",
    "cpsc_recent_value", "openfda_recent_value", "rank_delta", "regulatory_evidence_status",
]]
largest_rank_changes = inputs.reindex(
    inputs["rank_delta"].abs().sort_values(ascending=False).index
).head(15)[[
    "primary_ticker", "company_name", "sector", "rank_before_regulatory",
    "rank_after_regulatory", "rank_delta", "regulatory_evidence_penalty_provisional",
    "regulatory_evidence_status",
]]

# Overall coverage is the pillar-weighted share of observed evidence.
inputs["overall_observed_weight"] = 0.0
for prefix, title in PILLAR_FEATURE_KEYS.items():
    inputs["overall_observed_weight"] += FINAL_PILLAR_WEIGHTS[title] * inputs[f"{prefix}_observed_weight"]
for prefix in PILLAR_FEATURE_KEYS:
    inputs[f"{prefix}_raw_score"] = inputs[f"{prefix}_raw"]
    inputs[f"{prefix}_adjusted_score"] = inputs[f"{prefix}_adjusted"]
inputs["overall_coverage_grade"] = inputs["overall_observed_weight"].map(lambda value: esg.coverage_grade(value, CONFIDENCE_THRESHOLDS))
inputs["confidence_status"] = inputs["overall_coverage_grade"].map({
    "High": "High", "Medium": "Medium",
    "Low": "Low - diagnostic only; judge-facing ranking warning"})
inputs["quantitative_anchor_year"] = QUANTITATIVE_ANCHOR_YEAR
inputs["general_features_time_basis"] = GENERAL_FEATURES_TIME_BASIS
inputs["general_features_as_of_date"] = GENERAL_FEATURES_AS_OF_DATE
inputs["general_features_as_of_date_basis"] = GENERAL_FEATURES_AS_OF_DATE_BASIS
inputs["scoring_as_of_date"] = SCORING_AS_OF_DATE
inputs["assessment_description"] = ASSESSMENT_DESCRIPTION


In [ ]:
# Keep the main score fields first; diagnostics follow in a second group.
required_output_columns = [
    "company_id", "primary_ticker", "company_name", "sector", "quantitative_anchor_year",
    "general_features_time_basis", "general_features_as_of_date", "scoring_as_of_date",
    "environmental_raw_score", "environmental_adjusted_score", "environmental_observed_weight",
    "transition_raw_score", "transition_adjusted_score", "transition_observed_weight",
    "social_raw_score", "social_adjusted_score", "social_observed_weight",
    "governance_raw_score", "governance_adjusted_score", "governance_observed_weight",
    "structured_score_before_regulatory_penalty", "regulatory_evidence_penalty_provisional",
    "structured_score_after_regulatory_penalty", "net_zero_transition_score", "commitment_score",
    "realized_transition_score", "credibility_gap", "overall_observed_weight",
    "overall_coverage_grade", "confidence_status", "rank_before_regulatory",
    "rank_after_regulatory", "rank_delta",
]
extra_output_columns = [
    "constituent_tickers", "security_count", "assessment_description",
    "general_features_as_of_date_basis", "environmental_confidence",
    "transition_confidence", "social_confidence", "governance_confidence",
    "net_zero_transition_raw_score", "net_zero_transition_observed_weight",
    "net_zero_transition_confidence", "commitment_observed_weight", "commitment_confidence",
    "realized_transition_observed_weight", "realized_transition_confidence",
    "regulatory_penalty_observed_weight", "regulatory_evidence_status",
    "emissions_transition_classification", "scope12_absolute_trend_pct_per_year",
    "scope12_employee_intensity_trend_pct_per_year",
]
# One final company table is used by all exports and checks below.
scores = inputs[required_output_columns + extra_output_columns].sort_values("company_id").reset_index(drop=True)


## 5. Test sensitivity and inspect results

Recalculate the same score under alternative assumptions, then inspect leading, trailing, and unusual observations.


In [ ]:
def score_scenario(
    name,
    env_lambda=ENVIRONMENT_SECTOR_MIX_LAMBDA,
    pillar_weights=None,
    feature_weights=None,
    materiality_mode="baseline",
):
    # Start from the same normalized inputs used by the baseline score.
    frame = inputs.copy()
    sector_component = env_lambda * frame["scope12_reported_amount_2024_sector_score"]
    global_component = (1 - env_lambda) * frame["scope12_reported_amount_2024_global_score"]
    frame["scope12_reported_amount_2024_score"] = sector_component + global_component

    # Copy the baseline weights before replacing only the requested alternatives.
    weights = {}
    for pillar, values in FINAL_FEATURE_WEIGHTS.items():
        weights[pillar] = values.copy()
    for pillar, values in (feature_weights or {}).items():
        weights[pillar] = values.copy()

    materiality = FEATURE_MATERIALITY
    if materiality_mode == "flat":
        materiality = {}
        for feature in FEATURE_MATERIALITY:
            materiality[feature] = {sector: 1.0 for sector in SECTORS}
    selected_pillar_weights = (pillar_weights or FINAL_PILLAR_WEIGHTS).copy()

    # Recalculate each pillar explicitly so the scenario path mirrors the baseline.
    results = {}
    for prefix, title in PILLAR_FEATURE_KEYS.items():
        results[prefix] = esg.score_block(
            frame, weights[title], materiality,
            neutral_prior=NEUTRAL_PRIOR, confidence_thresholds=CONFIDENCE_THRESHOLDS)

    score_before_penalty = 0
    for prefix, title in PILLAR_FEATURE_KEYS.items():
        score_before_penalty += selected_pillar_weights[title] * results[prefix]["adjusted"]
    score_after_penalty = np.maximum(0, score_before_penalty - frame["regulatory_evidence_penalty_provisional"])
    scenario_rank = pd.Series(score_after_penalty, index=frame.index).rank(method="min", ascending=False).astype(int)

    return pd.DataFrame({
        "company_id": frame["company_id"],
        "scenario": name,
        "scenario_score": score_after_penalty,
        "scenario_rank": scenario_rank})


In [ ]:
# The final scenario changes feature weights; the others alter one broader assumption.
alternative_feature_weights = {
    "Transition": {
        "scope12_employee_intensity_trend": 0.40,
        "sbti_status": 0.40,
        "climate_governance_support": 0.20},
    "Governance": {
        "board_independence": 0.45, "ceo_separation": 0.15, "board_attendance": 0.15,
        "women_executives": 0.15, "sustainability_committee": 0.10},
}
scenario_definitions = [
    ("baseline", {}),
    ("pillar_environment_transition", {
        "pillar_weights": {"Environmental": 0.50, "Transition": 0.20, "Social": 0.15, "Governance": 0.15}}),
    ("pillar_social_governance", {
        "pillar_weights": {"Environmental": 0.35, "Transition": 0.10, "Social": 0.275, "Governance": 0.275}}),
    ("environment_lambda_0.25", {"env_lambda": 0.25}),
    ("environment_lambda_0.75", {"env_lambda": 0.75}),
    ("materiality_flat", {"materiality_mode": "flat"}),
    ("alternative_feature_weights", {"feature_weights": alternative_feature_weights}),
]

# Run the definitions through the same function and stack the results.
scenario_frames = []
for scenario_name, scenario_options in scenario_definitions:
    scenario_frames.append(score_scenario(scenario_name, **scenario_options))
sensitivity = pd.concat(scenario_frames, ignore_index=True)

# Attach the baseline score and rank to every scenario row.
baseline = sensitivity[sensitivity["scenario"].eq("baseline")]
baseline = baseline[["company_id", "scenario_score", "scenario_rank"]]
baseline = baseline.rename(columns={
    "scenario_score": "baseline_score",
    "scenario_rank": "baseline_rank"})
sensitivity = sensitivity.merge(baseline, on="company_id", how="left", validate="many_to_one")
sensitivity["rank_change_vs_baseline"] = sensitivity["baseline_rank"] - sensitivity["scenario_rank"]

# Summarize both score correlation and the size of rank changes.
scenario_correlations = {}
summary_rows = []
for scenario, group in sensitivity.groupby("scenario", sort=False):
    correlation = float(group["scenario_score"].corr(group["baseline_score"], method="spearman"))
    scenario_correlations[scenario] = correlation
    summary_rows.append({
        "scenario": scenario,
        "spearman_with_baseline": correlation,
        "maximum_absolute_rank_change": int(group["rank_change_vs_baseline"].abs().max()),
        "mean_absolute_rank_change": float(group["rank_change_vs_baseline"].abs().mean())})
sensitivity["spearman_with_baseline"] = sensitivity["scenario"].map(scenario_correlations)
company_labels = companies[["company_id", "primary_ticker", "company_name", "sector"]]
sensitivity = sensitivity.merge(company_labels, on="company_id", how="left", validate="many_to_one")
sensitivity_summary = pd.DataFrame(summary_rows)
display(sensitivity_summary.round(3))


In [ ]:
# Produce compact tables for presentation and a wider table for manual review.
top_bottom, manual_inspection = esg.build_inspection_tables(inputs, scores, PILLAR_FEATURE_KEYS, largest_penalties)
review_columns = ["table_position", "primary_ticker", "company_name", "structured_score_after_regulatory_penalty",
                  "net_zero_transition_score", "overall_coverage_grade", "component_explanation", "confidence_warning"]
display(top_bottom[review_columns])


## 6. Create outputs

Create the diagnostic figures and export the score, audit, coverage, and sensitivity tables.


In [ ]:
# Plotting details live in the module; the analytical inputs are explicit here.
representative, coverage_heatmap = esg.create_figures(inputs, OUTPUT_DIR, SECTORS, FINAL_PILLAR_WEIGHTS,
                                                         largest_rank_changes, top_bottom)
print("Representative company:", representative["primary_ticker"], representative["company_name"])
print("Figures created:", len(list(OUTPUT_DIR.glob("*.png"))))


In [ ]:
# Expand the nested weight dictionaries into one auditable table.
weight_rows = []
feature_weight_sets = {
    "original": ORIGINAL_FEATURE_WEIGHTS,
    "final": FINAL_FEATURE_WEIGHTS,
}
for weight_set, blocks in feature_weight_sets.items():
    for block, weights in blocks.items():
        for feature, weight in weights.items():
            weight_rows.append({
                "block": block, "weight_set": weight_set,
                "feature": feature, "weight": weight})
for weight_set, weights in {
    "original": ORIGINAL_NET_ZERO_WEIGHTS,
    "final": FINAL_NET_ZERO_WEIGHTS,
}.items():
    for feature, weight in weights.items():
        weight_rows.append({
            "block": "Net-zero transition", "weight_set": weight_set,
            "feature": feature, "weight": weight})
for pillar, weight in FINAL_PILLAR_WEIGHTS.items():
    weight_rows.append({
        "block": "Structured pillar blend", "weight_set": "final",
        "feature": pillar, "weight": weight})
weight_audit = pd.DataFrame(weight_rows)

# Expand both materiality matrices using the same simple loop.
materiality_rows = []
materiality_matrices = {
    "feature": FEATURE_MATERIALITY_LEVELS,
    "regulatory": REGULATORY_MATERIALITY_LEVELS,
}
for matrix_name, matrix in materiality_matrices.items():
    for feature_or_source, sector_levels in matrix.items():
        for sector, level in sector_levels.items():
            materiality_rows.append({
                "matrix": matrix_name, "feature_or_source": feature_or_source,
                "sector": sector, "level": level, "multiplier": MATERIALITY_VALUES[level]})
materiality_table = pd.DataFrame(materiality_rows)

# Every table is named here so the output folder is easy to audit.
exports = {
    "structured_score_coverage_by_sector_component.csv": coverage_heatmap.reset_index(names="sector"),
    "structured_score_feature_coverage_by_sector.csv": feature_coverage_by_sector,
    "structured_score_peer_fallbacks.csv": peer_diagnostics,
    "structured_score_manual_inspection.csv": manual_inspection,
    "structured_score_weight_audit.csv": weight_audit,
    "structured_score_materiality_matrix.csv": materiality_table,
    "structured_score_top_bottom_explanations.csv": top_bottom,
    "structured_score_sensitivity.csv": sensitivity,
    "structured_score_inputs_500.csv": inputs.sort_values("company_id"),
    "structured_scores_500.csv": scores,
    "structured_score_data_dictionary.csv": feature_dictionary,
}
for filename, table in exports.items():
    table.to_csv(OUTPUT_DIR / filename, index=False)
correlations.to_csv(OUTPUT_DIR / "structured_score_feature_spearman_correlations.csv")


## 7. Validate

Run the final checks once, after the complete score has been assembled.


In [ ]:
# Check score ranges, weights, coverage, and scenario consistency in one place.
score_columns = [
    "environmental_raw_score", "environmental_adjusted_score",
    "transition_raw_score", "transition_adjusted_score",
    "social_raw_score", "social_adjusted_score",
    "governance_raw_score", "governance_adjusted_score",
    "structured_score_before_regulatory_penalty", "structured_score_after_regulatory_penalty",
    "net_zero_transition_score", "commitment_score", "realized_transition_score",
]
# Record the sums as well as testing them so the validation file is informative.
configured_weight_sums = {
    "final_pillars": sum(FINAL_PILLAR_WEIGHTS.values()),
    **{f"final_{block.lower()}": sum(weights.values()) for block, weights in FINAL_FEATURE_WEIGHTS.items()},
    "final_net_zero": sum(FINAL_NET_ZERO_WEIGHTS.values()),
    "commitment": sum(COMMITMENT_WEIGHTS.values()),
    "realized_transition": sum(REALIZED_TRANSITION_WEIGHTS.values()),
    "regulatory_sources": sum(REGULATORY_SOURCE_WEIGHTS.values()),
}
coverage_columns = [
    *[f"{prefix}_observed_weight" for prefix in PILLAR_FEATURE_KEYS],
    "overall_observed_weight",
]
baseline_rows = sensitivity[sensitivity["scenario"].eq("baseline")].sort_values("company_id")
baseline_scores = baseline_rows["scenario_score"].to_numpy()
actual_scores = scores.sort_values("company_id")["structured_score_after_regulatory_penalty"].to_numpy()
forbidden_ivv_columns = [column for column in inputs if column in {"ivv_market_value_usd", "ivv_weight_pct"}]

# These are the conditions that would materially affect interpretation or ranking.
required_validation = {
    "exactly_500_unique_company_rows": len(scores) == 500 and scores["company_id"].nunique() == 500,
    "all_503_bloomberg_securities_mapped": len(general_mapped) == 503 and general_mapped["company_id"].notna().all(),
    "share_class_values_agree": not share_class_conflicts,
    "panel_has_6500_unique_company_year_rows": len(panel) == 6500 and not panel.duplicated(["company_id", "observation_year"]).any(),
    "panel_covers_2012_2024": sorted(panel["observation_year"].unique()) == list(range(2012, 2025)),
    "missing_and_zero_are_distinct": esg.clean_bloomberg_value(0) == 0 and pd.isna(esg.clean_bloomberg_value("#N/A N/A")),
    "ivv_market_value_is_unused": not forbidden_ivv_columns,
    "time_labels_are_correct": scores["general_features_time_basis"].eq(GENERAL_FEATURES_TIME_BASIS).all() and scores["quantitative_anchor_year"].eq(2024).all(),
    "scores_are_within_0_100": all(scores[column].dropna().between(0, 100).all() for column in score_columns),
    "regulatory_penalty_is_within_0_15": scores["regulatory_evidence_penalty_provisional"].between(0, 15).all(),
    "configured_weights_sum_to_one": all(np.isclose(value, 1.0) for value in configured_weight_sums.values()),
    "observed_weights_are_within_0_1": all(inputs[column].between(0, 1).all() for column in coverage_columns),
    "sustainability_and_net_zero_scores_are_separate": not np.allclose(scores["structured_score_after_regulatory_penalty"], scores["net_zero_transition_score"]),
    "baseline_scenario_matches_final_score": np.allclose(baseline_scores, actual_scores, equal_nan=True),
}
if not all(required_validation.values()):
    raise AssertionError({name: passed for name, passed in required_validation.items() if not passed})


In [ ]:
# Combine provenance, audit summaries, and final checks in one JSON record.
validation.update({
    "feature_summary": {
        "disabled_features": feature_dictionary.loc[
            feature_dictionary["feature_status"].str.startswith("Disabled"),
            ["output_feature", "decision_reason"],
        ].to_dict(orient="records"),
        "redundant_pairs": redundant_pairs,
        "peer_fallback_summary": peer_diagnostics.to_dict(orient="records")},
    "score_summary": {
        "pillar_summary": pillar_summary.to_dict(orient="records"),
        "configured_weight_sums": configured_weight_sums,
        "coverage_grades": scores["overall_coverage_grade"].value_counts().to_dict()},
    "regulatory_summary": {
        "penalty_min": float(inputs["regulatory_evidence_penalty_provisional"].min()),
        "penalty_max": float(inputs["regulatory_evidence_penalty_provisional"].max()),
        "companies_with_observed_evidence": int(inputs["regulatory_penalty_observed_weight"].gt(0).sum()),
        "largest_rank_changes": largest_rank_changes.head(5).to_dict(orient="records")},
    "sensitivity_summary": sensitivity_summary.to_dict(orient="records"),
    "required_validation": required_validation})
with open(OUTPUT_DIR / "structured_score_validation.json", "w", encoding="utf-8") as stream:
    json.dump(esg.json_safe(validation), stream, indent=2, default=str, allow_nan=False)

# Confirm that every principal artifact was written and is non-empty.
required_files = [
    "structured_score_inputs_500.csv", "structured_scores_500.csv",
    "structured_score_data_dictionary.csv", "structured_score_validation.json",
    "structured_score_sensitivity.csv", "01_score_decomposition_representative_company.png",
    "02_coverage_heatmap_by_sector_component.png",
    "03_sustainability_vs_net_zero_transition.png", "04_regulatory_rank_shift.png",
    "05_commitments_vs_realized_outcomes.png", "06_top_bottom_structured_scores.png",
]
artifact_check = pd.DataFrame([
    {
        "file": filename,
        "exists": (OUTPUT_DIR / filename).exists(),
        "bytes": (OUTPUT_DIR / filename).stat().st_size
        if (OUTPUT_DIR / filename).exists() else 0,
    }
    for filename in required_files])
assert artifact_check["exists"].all() and artifact_check["bytes"].gt(0).all()

passed_checks = sum(required_validation.values())
print(f"Complete: {len(scores)} company scores, {len(sensitivity):,} sensitivity rows, and {passed_checks} checks passed.")
